# 02 - Validacao das Integras TXT

Objetivo: validar a ligacao entre os metadados JSON e os arquivos TXT no ZIP usando `SeqDocumento`, sem fazer ainda analise textual substantiva.

Esta etapa tambem pode montar uma amostra pequena com texto integral para alimentar o notebook seguinte.


## 1. Ambiente

No Colab, clone o repositorio antes de abrir/rodar o notebook, ou abra o notebook diretamente pelo GitHub. Os dados brutos devem ficar no Google Drive, fora do Git.

In [1]:
# Rode esta celula no Colab se as dependencias ainda nao estiverem instaladas.
# !pip install -r requirements.txt

In [2]:
from pathlib import Path
import json
import re
import zipfile

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [3]:
from pathlib import Path
from google.colab import drive

MOUNT_POINT = Path("/content/drive")

if (MOUNT_POINT / "MyDrive").exists():
    print("Google Drive já está montado.")
else:
    drive.mount(str(MOUNT_POINT), force_remount=True)

DRIVE_DATA = Path("/content/drive/MyDrive/Mestrado/2026/llms/data")
RAW_STJ = DRIVE_DATA / "raw" / "stj_integras"
INTERIM_DATA = DRIVE_DATA / "interim"
PROCESSED_DATA = DRIVE_DATA / "processed"
REPORTS_DATA = DRIVE_DATA / "reports" / "summaries"

for path in [RAW_STJ, INTERIM_DATA, PROCESSED_DATA, REPORTS_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print("RAW_STJ:", RAW_STJ)
print("Arquivos encontrados:")
for file in sorted(RAW_STJ.iterdir()):
    print("-", file.name)


Google Drive já está montado.
RAW_STJ: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras
Arquivos encontrados:
- dicionariointegrasdecisoes.csv
- dicionariointegrasdecisoes.gsheet
- metadados20260319.json
- textos20260319.zip


## 2. Inspecao dos arquivos brutos

Esperado em `data/raw/stj_integras/` no Drive:

- um JSON de metadados;
- um ZIP com textos integrais em TXT;
- um CSV com dicionario de dados.

In [4]:
def inspect_raw_folder(raw_dir: str | Path) -> dict[str, list[Path]]:
    raw_dir = Path(raw_dir)
    files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
    return {
        'json': [p for p in files if p.suffix.lower() == '.json'],
        'zip': [p for p in files if p.suffix.lower() == '.zip'],
        'csv': [p for p in files if p.suffix.lower() == '.csv'],
        'other': [p for p in files if p.suffix.lower() not in {'.json', '.zip', '.csv'}],
    }

raw_files = inspect_raw_folder(RAW_STJ)
for kind, files in raw_files.items():
    print(kind.upper())
    for file in files:
        print(' ', file.name)

JSON
  metadados20260319.json
ZIP
  textos20260319.zip
CSV
  dicionariointegrasdecisoes.csv
OTHER
  dicionariointegrasdecisoes.gsheet


In [5]:
# Ajuste manualmente caso a pasta tenha mais de um arquivo do mesmo tipo.
METADATA_PATH = raw_files["json"][0] if raw_files["json"] else None
ZIP_PATH = raw_files["zip"][0] if raw_files["zip"] else None
DICTIONARY_PATH = raw_files["csv"][0] if raw_files["csv"] else None

if METADATA_PATH is None:
    raise FileNotFoundError(f"Nenhum arquivo .json encontrado em {RAW_STJ}")
if ZIP_PATH is None:
    raise FileNotFoundError(f"Nenhum arquivo .zip encontrado em {RAW_STJ}")

print("METADATA_PATH:", METADATA_PATH)
print("ZIP_PATH:", ZIP_PATH)
print("DICTIONARY_PATH:", DICTIONARY_PATH)


METADATA_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/metadados20260319.json
ZIP_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/textos20260319.zip
DICTIONARY_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras/dicionariointegrasdecisoes.csv


## 3. Leitura dos metadados

In [6]:
def load_metadata(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        list_values = [v for v in data.values() if isinstance(v, list)]
        if len(list_values) == 1:
            df = pd.DataFrame(list_values[0])
        else:
            df = pd.json_normalize(data)
    else:
        raise ValueError(f'Formato JSON inesperado: {type(data)}')

    print(f'Metadados: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
    return df

metadata_df = load_metadata(METADATA_PATH)
metadata_df.head()

Metadados: 5,968 linhas x 12 colunas


,SeqDocumento,dataPublicacao,tipoDocumento,numeroRegistro,processo,dataRecebimento,dataDistribuição,NM_MINISTRO,recurso,teor,descricaoMonocratica,assuntos
0,364440816,2026-03-19,ACÓRDÃO,202501710420,AREsp 2935515,2025-05-14,2025-05-19,PAULO SÉRGIO DOMINGUES,AgInt,Negando,None,"00014.05916.05954., 00014.05986.06007."
1,363279093,2026-03-19,DECISÃO,202503694349,AREsp 3060839,2025-09-24,2025-10-06,LUÍS CARLOS GAMBOGI (DESEMBARGADOR CONVOCADO D...,None,Negando,Conheço do agravo de #{nome_da_parte} para neg...,"01156.06220.07768., 00899.09616.05724.04939."
2,364320483,2026-03-19,ACÓRDÃO,202503847577,AREsp 3068328,2025-10-02,2025-10-24,JOÃO OTÁVIO DE NORONHA,None,Não Conhecendo,None,"00899.07681.07691.07698., 08826.08960.08961., ..."
3,363885758,2026-03-19,DECISÃO,202504253543,AREsp 3094586,2025-10-29,2025-11-12,NANCY ANDRIGHI,None,Negando,Conheço do agravo de #{nome_da_parte} para con...,"00899.10432.10448.10470., 00899.10432.10496."
4,364313111,2026-03-19,ACÓRDÃO,202402187446,AREsp 2669936,2024-06-17,2024-06-25,RICARDO VILLAS BÔAS CUEVA,None,Negando,None,00899.07681.07717.04968.


In [7]:
metadata_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5968 entries, 0 to 5967
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   SeqDocumento          5968 non-null   int64 
 1   dataPublicacao        5968 non-null   object
 2   tipoDocumento         5968 non-null   object
 3   numeroRegistro        5968 non-null   object
 4   processo              5968 non-null   object
 5   dataRecebimento       5968 non-null   object
 6   dataDistribuição      5968 non-null   object
 7   NM_MINISTRO           5968 non-null   object
 8   recurso               2536 non-null   object
 9   teor                  5968 non-null   object
 10  descricaoMonocratica  2367 non-null   object
 11  assuntos              5946 non-null   object
dtypes: int64(1), object(11)
memory usage: 559.6+ KB


In [8]:
metadata_df.columns.tolist()

['SeqDocumento',
 'dataPublicacao',
 'tipoDocumento',
 'numeroRegistro',
 'processo',
 'dataRecebimento',
 'dataDistribuição',
 'NM_MINISTRO',
 'recurso',
 'teor',
 'descricaoMonocratica',
 'assuntos']

## 4. Validacao do ZIP e da chave `SeqDocumento`

A validacao compara os `SeqDocumento` presentes nos metadados com os nomes dos arquivos TXT dentro do ZIP.


In [9]:
def list_zip_txt_files(zip_path: str | Path) -> list[str]:
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        return sorted([
            name for name in zf.namelist()
            if not name.endswith('/') and name.lower().endswith('.txt')
        ])

txt_files = list_zip_txt_files(ZIP_PATH)
print(f'TXT no ZIP: {len(txt_files):,}')
txt_files[:10]

TXT no ZIP: 497


['324789892.txt',
 '326163717.txt',
 '326498583.txt',
 '349720200.txt',
 '349753353.txt',
 '352275673.txt',
 '354661883.txt',
 '354709385.txt',
 '354994634.txt',
 '356385294.txt']

## 6. Validacao da chave `SeqDocumento`

In [10]:
def normalize_seq_documento(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text.endswith('.0'):
        text = text[:-2]
    return text

def txt_stem(name: str) -> str:
    return Path(name).stem.strip()

def validate_seqdocumento_link(metadata_df: pd.DataFrame, txt_files: list[str]) -> dict:
    if 'SeqDocumento' not in metadata_df.columns:
        raise KeyError('Coluna SeqDocumento nao encontrada nos metadados')

    metadata_keys = set(
        metadata_df['SeqDocumento']
        .map(normalize_seq_documento)
        .dropna()
    )
    txt_keys = {txt_stem(name) for name in txt_files}

    matched = metadata_keys & txt_keys
    metadata_without_txt = metadata_keys - txt_keys
    txt_without_metadata = txt_keys - metadata_keys

    return {
        'total_metadata_rows': len(metadata_df),
        'total_metadata_keys': len(metadata_keys),
        'total_txt_files': len(txt_files),
        'matched_keys': len(matched),
        'metadata_without_txt': len(metadata_without_txt),
        'txt_without_metadata': len(txt_without_metadata),
        'metadata_without_txt_examples': sorted(metadata_without_txt)[:20],
        'txt_without_metadata_examples': sorted(txt_without_metadata)[:20],
    }

validation_report = validate_seqdocumento_link(metadata_df, txt_files)
validation_report

{'total_metadata_rows': 5968,
 'total_metadata_keys': 5968,
 'total_txt_files': 497,
 'matched_keys': 497,
 'metadata_without_txt': 5471,
 'txt_without_metadata': 0,
 'metadata_without_txt_examples': ['262555859',
  '283635738',
  '283650216',
  '296107529',
  '308766004',
  '314455964',
  '314458031',
  '314699367',
  '320627185',
  '321588285',
  '325456354',
  '325639374',
  '329171740',
  '329493596',
  '329599467',
  '331228839',
  '332331698',
  '334652405',
  '335177186',
  '335865607'],
 'txt_without_metadata_examples': []}

In [11]:
validation_path = REPORTS_DATA / 'stj_integras_key_validation.json'
validation_path.write_text(json.dumps(validation_report, ensure_ascii=False, indent=2), encoding='utf-8')
validation_path

PosixPath('/content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_integras_key_validation.json')

## 7. Limpeza textual basica

In [12]:
def clean_legal_text(text: str) -> str:
    if text is None or pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'<\s*br\s*/?\s*>', '\n', text, flags=re.IGNORECASE)
    text = BeautifulSoup(text, 'html.parser').get_text(' ')
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text.strip()

clean_legal_text('Linha 1<br>Linha 2<br/> <b>Texto</b> juridico.')

'Linha 1\nLinha 2\n Texto juridico.'

## 8. Construcao de corpus amostral

In [13]:
EXPECTED_COLUMNS = [
    'SeqDocumento',
    'dataPublicacao',
    'tipoDocumento',
    'processo',
    'ministro',
    'NM_MINISTRO',
    'teor',
    'descricaoMonocratica',
    'assuntos',
]

def read_txt_from_zip(zf: zipfile.ZipFile, file_name: str) -> str:
    raw = zf.read(file_name)
    for encoding in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode('utf-8', errors='replace')

def build_sample_corpus(metadata_df: pd.DataFrame, zip_path: str | Path, n: int = 50, random_state: int = 42) -> pd.DataFrame:
    zip_path = Path(zip_path)
    df = metadata_df.copy()
    df['_seq_key'] = df['SeqDocumento'].map(normalize_seq_documento)

    with zipfile.ZipFile(zip_path) as zf:
        txt_files = list_zip_txt_files(zip_path)
        txt_by_key = {txt_stem(name): name for name in txt_files}
        available = df[df['_seq_key'].isin(txt_by_key)].copy()

        sample_n = min(n, len(available))
        sample = available.sample(n=sample_n, random_state=random_state) if sample_n else available

        rows = []
        for _, row in tqdm(sample.iterrows(), total=len(sample)):
            key = row['_seq_key']
            texto_original = read_txt_from_zip(zf, txt_by_key[key])
            item = {col: row[col] if col in row.index else None for col in EXPECTED_COLUMNS}
            if not item.get('ministro') and item.get('NM_MINISTRO'):
                item['ministro'] = item['NM_MINISTRO']
            item['texto_original'] = texto_original
            item['texto_limpo'] = clean_legal_text(texto_original)
            rows.append(item)

    return pd.DataFrame(rows)

sample_corpus = build_sample_corpus(metadata_df, ZIP_PATH, n=50)
sample_corpus.head()

  0%|          | 0/50 [00:00<?, ?it/s]

,SeqDocumento,dataPublicacao,tipoDocumento,processo,ministro,NM_MINISTRO,teor,descricaoMonocratica,assuntos,texto_original,texto_limpo
0,362798716,2026-03-19,DECISÃO,HC 1078698,MARIA MARLUCE CALDAS,MARIA MARLUCE CALDAS,Negando,Denegado o Habeas Corpus de #{nome_da_parte} #...,00287.03603.03607.03608.,"DECISÃO<br>Trata-se de habeas corpus, com pedi...","DECISÃO\nTrata-se de habeas corpus, com pedido..."
1,360879614,2026-03-19,DECISÃO,HC 1057760,MARIA MARLUCE CALDAS,MARIA MARLUCE CALDAS,Concedendo,Não conhecido o Habeas Corpus. Concedido o Hab...,01209.07942.07791.,"DECISÃO<br>Trata-se de habeas corpus, com pedi...","DECISÃO\nTrata-se de habeas corpus, com pedido..."
2,364093156,2026-03-19,DECISÃO,HC 1079862,HERMAN BENJAMIN,HERMAN BENJAMIN,Negando,Denegado o Habeas Corpus de #{nome_da_parte} #...,00287.03415.03416.,DECISÃO<br>Cuida-se de Habeas Corpus impetrado...,DECISÃO\nCuida-se de Habeas Corpus impetrado e...
3,362865610,2026-03-19,DECISÃO,HC 1063902,SEBASTIÃO REIS JÚNIOR,SEBASTIÃO REIS JÚNIOR,Concedendo,Concedido o Habeas Corpus a #{nome_da_parte} #...,00287.03603.03607.03608.,DECISÃO<br>Trata-se de habeas corpus impetrado...,DECISÃO\nTrata-se de habeas corpus impetrado e...
4,363980237,2026-03-19,DECISÃO,HC 1080463,RIBEIRO DANTAS,RIBEIRO DANTAS,Não Conhecendo,Não conhecido o Habeas Corpus de #{nome_da_par...,00287.03415.05566.,DECISÃO<br>Trata-se de habeas corpus substitut...,DECISÃO\nTrata-se de habeas corpus substitutiv...


In [14]:
sample_corpus.assign(
    chars_original=sample_corpus['texto_original'].str.len(),
    chars_limpo=sample_corpus['texto_limpo'].str.len(),
)[['SeqDocumento', 'tipoDocumento', 'ministro', 'teor', 'chars_original', 'chars_limpo']].head(10)

,SeqDocumento,tipoDocumento,ministro,teor,chars_original,chars_limpo
0,362798716,DECISÃO,MARIA MARLUCE CALDAS,Negando,7917,7800
1,360879614,DECISÃO,MARIA MARLUCE CALDAS,Concedendo,12052,11917
2,364093156,DECISÃO,HERMAN BENJAMIN,Negando,10734,10629
3,362865610,DECISÃO,SEBASTIÃO REIS JÚNIOR,Concedendo,9049,8943
4,363980237,DECISÃO,RIBEIRO DANTAS,Não Conhecendo,12703,12562
5,364305370,DECISÃO,JOEL ILAN PACIORNIK,Concedendo,18364,18196
6,363547766,DECISÃO,OG FERNANDES,Outros,3598,3544
7,364282861,DECISÃO,CARLOS PIRES BRANDÃO,Negando,15520,15307
8,364136100,DECISÃO,RIBEIRO DANTAS,Negando,23115,22793
9,363748950,DECISÃO,REYNALDO SOARES DA FONSECA,Não Conhecendo,47255,46768


In [15]:
sample_parquet = PROCESSED_DATA / 'stj_integras_sample.parquet'
sample_csv = PROCESSED_DATA / 'stj_integras_sample.csv'

sample_corpus.to_parquet(sample_parquet, index=False)
sample_corpus.to_csv(sample_csv, index=False)

print('Salvo em:')
print('-', sample_parquet)
print('-', sample_csv)

Salvo em:
- /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_integras_sample.parquet
- /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_integras_sample.csv


## 8. Proximos passos

- Conferir `stj_integras_key_validation.json`.
- Se a cobertura da chave estiver boa, usar a amostra salva em `processed/` no notebook `03_analise_textual_inicial.ipynb`.
- Se a cobertura estiver baixa, verificar se o JSON e o ZIP sao do mesmo pacote/data.
